# Machine Learning I — Predicting Returns with Many Characteristics
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Frame return prediction as a regression problem** with many characteristics
2. **Apply Lasso, Ridge, and Elastic Net** for feature selection and shrinkage
3. **Use cross-validation** for hyperparameter tuning without lookahead
4. **Compare ML predictions to the OLS baseline**
5. **Audit AI-generated ML code** — train/test split timing, leakage, hyperparam grids

## 📋 TOC
1. [Setup](#setup)  2. [The ML Problem in Finance](#problem)
3. [Pitfall Checklist](#pitfalls)  4. [Lasso, Ridge, Elastic Net](#linear)
5. [Cross-Validation Without Lookahead](#cv)  6. [🎯 Challenge: Tune a Predictor](#challenge)
7. [Submission](#submit)  8. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## The ML Problem in Finance <a id="problem"></a>

You have $N$ stocks, each with $K$ characteristics (B/M, momentum, profitability,
size, idio vol, accruals, ...). You want to predict next-month return.

$$r_{i,t+1} = f(X_{i,t}^{(1)}, X_{i,t}^{(2)}, \ldots, X_{i,t}^{(K)}) + \epsilon_{i,t+1}$$

With $K = 100+$ characteristics and $T = 240$ months, naive OLS overfits.
ML methods exist to shrink or select among the K coefficients.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Standard CV with time-series data** | Random k-fold contaminates train with test | Use **walk-forward** or **time-series CV** |
| 2 | **Leakage in feature normalization** | Z-scoring with full-sample mean/std uses future info | Normalize within each cross-section |
| 3 | **Y included in feature engineering** | Computing target-encoded features uses Y → leakage | Audit every feature for Y dependency |
| 4 | **Hyperparameter overfit** | Tuning over 1000 grid points eventually picks one that "works" by chance | Hold out a final test set never seen during tuning |
| 5 | **Reporting in-sample R²** | OLS / Lasso R² always positive. The OOS R² is what matters and can be negative. | Always report OOS metrics |
| 6 | **Sklearn API has surprising defaults** | `Lasso(alpha=1)` is very different from `LassoCV()` | Read the doc; check selected alpha vs the grid endpoints |

---
## Lasso, Ridge, Elastic Net <a id="linear"></a>

Three penalized regressions:

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge** ($L_2$) | $\lambda \sum \beta_j^2$ | Shrinks all coefficients toward zero |
| **Lasso** ($L_1$) | $\lambda \sum |\beta_j|$ | Sets some coefficients exactly to zero (sparse) |
| **Elastic Net** | mix of both | Sparse + stable when features are correlated |

In return prediction, **elastic net** usually wins because financial
features are highly correlated (book-to-market, earnings-to-price, etc.
all measure similar things).

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
np.random.seed(42)

# Simulate: 50 features, only 3 actually matter
N, K = 500, 50
X = np.random.normal(0, 1, (N, K))
true_betas = np.zeros(K)
true_betas[[0, 5, 10]] = [0.2, -0.1, 0.15]
y = X @ true_betas + np.random.normal(0, 0.5, N)

# Train/test split
X_train, X_test = X[:400], X[400:]
y_train, y_test = y[:400], y[400:]

# OLS for comparison
ols = LinearRegression().fit(X_train, y_train)
ols_r2 = ols.score(X_test, y_test)

# Penalized methods
lasso = Lasso(alpha=0.05).fit(X_train, y_train)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
enet  = ElasticNet(alpha=0.05, l1_ratio=0.5).fit(X_train, y_train)

print(f"OOS R²:")
print(f"  OLS:         {ols.score(X_test, y_test):.3f}")
print(f"  Lasso:       {lasso.score(X_test, y_test):.3f}")
print(f"  Ridge:       {ridge.score(X_test, y_test):.3f}")
print(f"  Elastic Net: {enet.score(X_test, y_test):.3f}")
print(f"\nLasso selected {(lasso.coef_ != 0).sum()} of {K} features.")

---
## Cross-Validation Without Lookahead <a id="cv"></a>

**Standard k-fold CV is WRONG for time series** because it randomly splits
your data — meaning your training folds can contain *future* data relative
to your validation fold. This gives optimistic OOS estimates.

**Walk-forward CV** instead:
1. Train on months 1-60, validate on month 61
2. Train on months 1-61, validate on month 62
3. ... etc.

Or **time-series CV** with expanding/rolling windows. Sklearn has `TimeSeriesSplit`.

---
## 🎯 Challenge: Tune a Predictor <a id="challenge"></a>

> **Setup.** Using the simulated data above (X, y), find the best regularization
> parameter for Elastic Net via 5-fold CV.

### Q1 — Best alpha for Elastic Net

> **📌 Required:**
> ```python
> from sklearn.linear_model import ElasticNetCV
> enet_cv  = ElasticNetCV(alphas=____, l1_ratio=0.5, cv=5, max_iter=10000).fit(X_train, y_train)
> best_alpha = ____   # enet_cv.alpha_
> ```

In [ ]:
from sklearn.linear_model import ElasticNetCV
alphas_grid = np.logspace(-3, 0, 50)

enet_cv = ElasticNetCV(alphas=alphas_grid, l1_ratio=0.5, cv=5, max_iter=10000).fit(X_train, y_train)
best_alpha = ____    # extract from enet_cv
print(f"Best alpha: {best_alpha:.4f}")

### Q2 — OOS R² of tuned Elastic Net

> **📌 Required:**
> ```python
> oos_r2 = ____   # enet_cv.score(X_test, y_test)
> ```

In [ ]:
oos_r2 = ____
print(f"OOS R² of tuned ENet: {oos_r2:.3f}")

### Q3 — Feature selection

How many features did the tuned model select (non-zero coefficient)?

> **📌 Required:**
> ```python
> n_selected = ____    # (enet_cv.coef_ != 0).sum()
> ```

In [ ]:
n_selected = ____
print(f"Selected {n_selected} of {K} features. True nonzero = 3.")

### Q4 — Memo

Max 5 sentences. Comment on (i) whether ML beat OLS, (ii) whether the tuned
model recovered the 3 true features, (iii) what could go wrong if you applied
this to real return data instead of simulated.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["best_alpha", "oos_r2", "n_selected", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "ML_I_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Penalized regression beats OLS** when you have many correlated features.
2. **Elastic Net usually wins** for finance because features are highly correlated.
3. **Time-series CV is mandatory** for any backtest involving model tuning.
4. **OOS R² is the only honest metric.** In-sample numbers always look good.
5. **AI writes the sklearn boilerplate. You design the train/test split and audit for leakage.**